In [ ]:
from load_data_function import load_data,downsample_all_battery_cycle,fig_plot
from torch.utils.data import DataLoader, TensorDataset
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
from config import Config
import os

In [ ]:
def downsample_all_battery_cycle(dataset_path, new_length,dataset):
    """
    该函数主要用于对每个循环进行降采样，将其变为同一长度，便于后续输入网络
    :param dataset_path: E:\Battery_data\Data\Converted_UCSD Nissan
    :param new_length: 降采样后的长度
    :param threshold: 满足循环所设置的电压区间，用于保存文件名
    :return:
    """
    all_battery_id_data_path = f'{dataset_path}\\{dataset}_data.pkl'
    all_battery_id_soh_path = f'{dataset_path}\\{dataset}_SOH.pkl'
    #downsample_all_battery_id_data_path = f'{dataset_path}\\{package}\\_{new_length}_{config.All_Dataset[0]}_all_battery_id_data.pkl'
    with open(all_battery_id_data_path, 'rb') as file:
        data = pickle.load(file)
    with open(all_battery_id_soh_path, 'rb') as file:
        soh_data = pickle.load(file)
        for package in data.keys():
            battery_data= data[package]
            battery_soh = soh_data[package]
            downsample_all_battery_id_soh_path=f'{dataset_path}\\downsampled_data_{new_length}\\{package}\\{new_length}_{dataset}_all_battery_id_soh.pkl'
            downsample_all_battery_id_data_path =f'{dataset_path}\\downsampled_data_{new_length}\\{package}\\{new_length}_{dataset}_all_battery_id_data.pkl'
            os.makedirs(os.path.dirname(downsample_all_battery_id_data_path), exist_ok=True)
            new_data = {}
            os.makedirs(os.path.dirname(downsample_all_battery_id_soh_path), exist_ok=True)
            for battery_id in list(battery_data.keys()):
                new_data[battery_id] = []
                for cycle_i in range(len(battery_data[f'{battery_id}'])):
                    cycle_data = battery_data[f'{battery_id}'][cycle_i]
                    # 将时间第一个数改为0，所有数均减去第一个数
                    cycle_data[2] = cycle_data[2] - cycle_data[2][0]
                    # 原始数据的长度
                    original_length = cycle_data.shape[1]
                    # 原始数据的索引
                    original_indices = np.linspace(0, original_length - 1, original_length)
                    # 目标降采样后的索引
                    new_indices = np.linspace(0, original_length - 1, new_length)
                    # 对每一行进行线性插值降采样
                    cycle_data_resampled = np.zeros((cycle_data.shape[0], new_length))
                    for i in range(cycle_data.shape[0]):
                        cycle_data_resampled[i] = np.interp(new_indices, original_indices, cycle_data[i])
                    # 将降采样后的数据保存回新字典
                    new_data[battery_id].append(cycle_data_resampled)
                print(f'电池-{battery_id}已处理完成')
            print(f'电池组-{package}已处理完成')
            with open(downsample_all_battery_id_data_path, 'wb') as file:
                pickle.dump(new_data, file)
            with open(downsample_all_battery_id_soh_path, 'wb') as file:
                pickle.dump(battery_soh, file)

config = Config()
dataset_path = f'D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data/{config.All_Dataset[0]}_dataset'
new_length=256
downsample_all_battery_cycle(dataset_path, new_length, config.All_Dataset[0])

In [ ]:
for dataset in list(config.All_Dataset):
    dataset_path = f'D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data/{dataset}_dataset'
    new_length=256
    downsample_all_battery_cycle(dataset_path, new_length, dataset)
    print(f'{dataset}数据集降采样完成')

In [ ]:
package='package_1'
new_length=256
dataset='XJTU_battery'
config = Config()
dataset_path = f'D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data/{config.All_Dataset[10]}_dataset'

downsample_all_battery_id_soh_path=f'{dataset_path}\\downsampled_data_{new_length}\\{package}\\{new_length}_{dataset}_all_battery_id_soh.pkl'
downsample_all_battery_id_data_path =f'{dataset_path}\\downsampled_data_{new_length}\\{package}\\{new_length}_{dataset}_all_battery_id_data.pkl'
data=load_data(downsample_all_battery_id_data_path)
soh_data=load_data(downsample_all_battery_id_soh_path)
fig_plot(soh_data['battery_1'])


In [ ]:
print(soh_data.keys())
print(data.keys())
print(data['battery_1'][0].shape)

In [ ]:
fig_plot(data['battery_1'][0][2])